In [1]:
%%capture
!pip install --pre -U langchain langchain_core langchain-openai langsmith
!pip install langchain-huggingface langchain-qdrant qdrant-client transformers accelerate bitsandbytes

In [18]:
import torch
from qdrant_client import QdrantClient
from operator import itemgetter
from langchain_core.load import dumps, loads
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import datetime
import gc

from google.colab import userdata
import json
from tqdm import tqdm
import os

In [3]:
config = {
    'LANGSMITH_ENDPOINT': "https://api.smith.langchain.com",
    'LANGSMITH_PROJECT': "skripsi",
    'LANGSMITH_API_KEY': userdata.get('LANGSMITH_API_KEY'),
    'OPENAI_API_KEY': userdata.get('OPENAI_API_KEY'),
    'QDRANT_API_KEY': userdata.get('QDRANT_API_KEY'),
    'QDRANT_URL': userdata.get('QDRANT_URL'),
    'HF_TOKEN': userdata.get('HF_TOKEN'),
    'EMBEDDING_MODEL': "google/embeddinggemma-300m",
    'COLLECTION_NAME': "enriched_medical_reports_v1",
    'MODEL_ID': "Qwen/Qwen3-4B-Instruct-2507",
    'TOP_K': 3
    }

current_date = datetime.datetime.now().strftime("%Y%m%d_%H%M")
input_file = "nutrition_qa_golden_dataset.json"
output_file = f"{current_date}_rag_responses_dataset_k{config['TOP_K']}.jsonl"

In [ ]:
# SETUP ENVIRONMENT
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = config['LANGSMITH_ENDPOINT']
os.environ["LANGCHAIN_PROJECT"] = config['LANGSMITH_PROJECT']
os.environ["LANGCHAIN_API_KEY"] = config['LANGSMITH_API_KEY']
os.environ["OPENAI_API_KEY"] = config['OPENAI_API_KEY']

# SETUP VECTORSTORE & EMBEDDING
client = QdrantClient(url=config['QDRANT_URL'], api_key=config['QDRANT_API_KEY'])
embedding_model = HuggingFaceEmbeddings(
    model_name=config['EMBEDDING_MODEL'],
    model_kwargs={'device': 'cuda', 'token': config['HF_TOKEN']}
)
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=config['COLLECTION_NAME'],
    embedding=embedding_model,
    content_payload_key="page_content",
)
baseline_retriever = vectorstore.as_retriever(search_kwargs={"k": config['TOP_K']})
fusion_retriever = vectorstore.as_retriever(search_kwargs={"k": config['TOP_K'] * 3})

# 3. SETUP LLM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(config['MODEL_ID'])
model = AutoModelForCausalLM.from_pretrained(
    config['MODEL_ID'],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
text_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    repetition_penalty=1.1
)
shared_llm = HuggingFacePipeline(pipeline=text_pipe)

print("Semua model berhasil dimuat ke memori (Hanya 1 Salinan).")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Device set to use cuda:0


Semua model berhasil dimuat ke memori (Hanya 1 Salinan).


In [ ]:
class RAGBaseline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.setup_generation_chain()

    def setup_generation_chain(self):
        template = """<|im_start|>system
Anda adalah asisten medis profesional. Tugas Anda adalah menjawab pertanyaan klinis berdasarkan dokumen rekam medis pasien yang diberikan.

ATURAN MENJAWAB:
1. Ekstrak dan sintesis informasi dari KONTEKS untuk menjawab PERTANYAAN.
2. Jawab dengan komprehensif, logis, dan langsung pada intinya. Dilarang mengulang pertanyaan.
3. Anda diizinkan untuk merangkum atau menghubungkan beberapa fakta yang ada di dalam konteks.
4. Jika dan HANYA JIKA konteks sama sekali tidak membahas topik yang ditanyakan, barulah Anda menjawab "Informasi tidak ditemukan."
<|im_end|>
<|im_start|>user
KONTEKS:
{context}

PERTANYAAN:
{question}
<|im_end|>
<|im_start|>assistant
"""
        self.prompt = ChatPromptTemplate.from_template(template)
        self.generation_chain = (
            self.prompt | self.llm | StrOutputParser()
        )

    @staticmethod
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    def run(self, query):
        retrieved_docs = self.retriever.invoke(query)
        formatted_context = self.format_docs(retrieved_docs)
        answer = self.generation_chain.invoke({
            "context": formatted_context,
            "question": query
        })
        return {"answer": answer, "contexts": retrieved_docs}

class RAGFusion:
    def __init__(self, retriever, llm, top_n):
        self.retriever = retriever
        self.llm = llm
        self.setup_fusion_pipeline()
        self.setup_generation_chain()
        self.top_n = top_n

    def reciprocal_rank_fusion(self, results: list[list], k=60):
        fused_scores = {}
        for docs in results:
            for rank, doc in enumerate(docs):
                try:
                    doc_str = doc.json()
                except AttributeError:
                    doc_str = json.dumps(doc)

                if doc_str not in fused_scores:
                    fused_scores[doc_str] = 0
                fused_scores[doc_str] += 1 / (rank + k)

        reranked_results = [
            (json.loads(doc), score)
            for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        ]

        final_docs = []
        for doc_data, _ in reranked_results:
            if isinstance(doc_data, dict):
                final_docs.append(Document(
                    page_content=doc_data.get('page_content', ''),
                    metadata=doc_data.get('metadata', {})
                ))
            else:
                final_docs.append(doc_data)

        return final_docs[:self.top_n]

    def setup_fusion_pipeline(self):
        template_query_gen = """<|im_start|>system
          Anda adalah asisten AI ahli dalam mengoptimalkan pencarian dokumen Rekam Medis (EHR).
          Tugas Anda adalah memecah informasi dari pengguna menjadi 3 kueri pencarian (search queries) yang spesifik, tajam, dan berbeda.

          ATURAN PEMBUATAN KUERI:
          1. Input pengguna berisi Diagnosis Pasien dan Pertanyaan Inti. Anda WAJIB menyertakan kata kunci diagnosis atau gejala utama di setiap kueri agar pencarian tidak melenceng ke pasien lain.
          2. Buat kueri dalam bentuk frasa kata kunci yang padat (keyword-focused), bukan kalimat tanya yang panjang.
          3. Gunakan variasi istilah medis atau sinonim yang relevan jika diperlukan.

          FORMAT WAJIB:
          - Tulis TEPAT 3 baris teks.
          - DILARANG menggunakan angka (1., 2., 3.), tanda hubung (-), atau bullet point di awal kalimat.
          - DILARANG menulis basa-basi pengantar atau penutup. Langsung berikan 3 baris kueri tersebut.
          <|im_end|>
          <|im_start|>user
          Informasi Pasien dan Pertanyaan:
          {question}
          <|im_end|>
          <|im_start|>assistant
        """
        prompt_rag_fusion = ChatPromptTemplate.from_template(template_query_gen)

        self.generate_queries = (
            prompt_rag_fusion
            | ChatOpenAI(temperature=0.2)
            | StrOutputParser()
            | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
        )

        self.retrieval_chain_rag_fusion = (
            self.generate_queries
            | self.retriever.map()
            | self.reciprocal_rank_fusion
        )

    def setup_generation_chain(self):
        template = """<|im_start|>system
          Anda adalah asisten medis profesional. Tugas Anda adalah menjawab pertanyaan klinis berdasarkan dokumen rekam medis pasien yang diberikan.

          ATURAN MENJAWAB:
          1. Ekstrak dan sintesis informasi dari KONTEKS untuk menjawab PERTANYAAN.
          2. Jawab dengan komprehensif, logis, dan langsung pada intinya. Dilarang mengulang pertanyaan.
          3. Anda diizinkan untuk merangkum atau menghubungkan beberapa fakta yang ada di dalam konteks.
          4. Jika dan HANYA JIKA konteks sama sekali tidak membahas topik yang ditanyakan, barulah Anda menjawab "Informasi tidak ditemukan."
          <|im_end|>
          <|im_start|>user
          KONTEKS:
          {context}

          PERTANYAAN:
          {question}
          <|im_end|>
          <|im_start|>assistant
        """
        self.prompt = ChatPromptTemplate.from_template(template)
        self.generation_chain = (
            self.prompt | self.llm | StrOutputParser()
        )

    @staticmethod
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    def run(self, query):
        fused_docs = self.retrieval_chain_rag_fusion.invoke({"question": query})
        formatted_context = self.format_docs(fused_docs)
        answer = self.generation_chain.invoke({
            "context": formatted_context,
            "question": query
        })
        return {"answer": answer, "contexts": fused_docs}

In [ ]:
print("Menginisialisasi Model dan Vector Database...")
rag_baseline = RAGBaseline(retriever=shared_retriever, llm=shared_llm)
rag_fusion = RAGFusion(retriever=shared_retriever, llm=shared_llm, top_n=config['TOP_K'])
print("Inisialisasi selesai!\n")

Menginisialisasi Model dan Vector Database...
Inisialisasi selesai!



In [ ]:
with open(input_file, 'r', encoding='utf-8') as f:
        synthetic_dataset = json.load(f)

print(f"Memulai eksekusi untuk {len(synthetic_dataset)} pertanyaan...")

with open(output_file, 'a', encoding='utf-8') as f_out:
    for item in tqdm(synthetic_dataset, desc="Proses Evaluasi RAG"):
        original_question = item['question']
        first_chunk_meta = item['chunk_metadata'][0]
        anthro = first_chunk_meta.get('anthropometric_assessment', '')
        diagnosis = first_chunk_meta.get('medical_diagnosis', '')

        query_to_run = f"Pasien dengan {anthro} dan diagnosis medis {diagnosis}. {original_question}"

        try:
            baseline_res = rag_baseline.run(query_to_run)
            baseline_answer = baseline_res['answer']
            baseline_contexts = [doc.page_content for doc in baseline_res['contexts']]
            baseline_contexts_metadata = [doc.metadata for doc in baseline_res['contexts']]
        except Exception as e:
            print(f"\n[Error Baseline] Gagal memproses pertanyaan: '{original_question[:30]}...'\nError: {e}")
            baseline_answer, baseline_contexts = "ERROR", []

        try:
            fusion_res = rag_fusion.run(query_to_run)
            fusion_answer = fusion_res['answer']
            fusion_contexts = [doc.page_content for doc in fusion_res['contexts']]
            fusion_contexts_metadata = [doc.metadata for doc in fusion_res['contexts']]
        except Exception as e:
            print(f"\n[Error Fusion] Gagal memproses pertanyaan: '{original_question[:30]}...'\nError: {e}")
            fusion_answer, fusion_contexts = "ERROR", []

        result_item = {
            "question": original_question,
            "ground_truth": item['ground_truth'],
            "chunk_metadata": item['chunk_metadata'],

            "baseline_answer": baseline_answer,
            "baseline_contexts_metadata": baseline_contexts_metadata,
            "baseline_retrieved_contexts": baseline_contexts,

            "fusion_answer": fusion_answer,
            "fusion_contexts_metadata": fusion_contexts_metadata,
            "fusion_retrieved_contexts": fusion_contexts
        }

        f_out.write(json.dumps(result_item, ensure_ascii=False) + '\n')
        f_out.flush()

        if 'baseline_res' in locals(): del baseline_res
        if 'fusion_res' in locals(): del fusion_res

        gc.collect()

        torch.cuda.empty_cache()

print("\nEksekusi selesai! Semua data aman tersimpan di disk.")


Memulai eksekusi untuk 150 pertanyaan...


Proses Evaluasi RAG:   0%|          | 0/150 [00:00<?, ?it/s]/tmp/ipython-input-521/3030242716.py:57: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  doc_str = doc.json()
Proses Evaluasi RAG:  21%|██▏       | 32/150 [56:24<3:29:55, 106.74s/it]


[Error Fusion] Gagal memproses pertanyaan: 'Bagaimana hasil pemeriksaan bi...'
Error: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 393.81 MiB is free. Including non-PyTorch memory, this process has 14.18 GiB memory in use. Of the allocated memory 13.00 GiB is allocated by PyTorch, and 1.01 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Proses Evaluasi RAG:  71%|███████▏  | 107/150 [2:58:23<1:03:49, 89.07s/it]